In [4]:

import re
import json
import datetime
from collections import defaultdict

INPUT_FILE = "container_logs.txt"
# OUTPUT_FILE = "labeled_audit.json"
# WINDOW_OUTPUT_FILE = "labeled_windows.json"

# Parameters for windowing (tune for model)
WINDOW_SIZE = 30        # number of events in a window
WINDOW_STRIDE = 10      # hop size
# If you don't want windowing, set WINDOW_SIZE = 0

# Whitelisted benign container/daemon commands (common)
BENIGN_PROCS = {
    "systemd-resolve", "NetworkManager", "dnsmasq", "chronyd", "auditd",
    "containerd", "dockerd", "kubelet", "crio", "crun", "conmon", "systemd"
}

# Suspicious binary/name indicators (extend as needed)
SUSPICIOUS_BINARY_KEYWORDS = {
    "xmrig", "minerd", "minergate", "coinhive", "nc", "netcat", "socat", "bash", "sh"
}

# Helper: convert audit(...) epoch to ISO8601
def epoch_to_iso(epoch_str):
    try:
        # epoch may be float like 1762081984.956
        epoch = float(epoch_str)
        dt = datetime.datetime.utcfromtimestamp(epoch)
        # preserve milliseconds
        iso = dt.isoformat(timespec='milliseconds') + "Z"
        return iso
    except:
        return None

def parse_numeric_fields(line):
    numeric = {}
    for field in ["pid", "ppid", "syscall", "success", "uid", "auid", "gid"]:
        m = re.search(rf'{field}=([0-9]+)', line)
        if m:
            numeric[field] = int(m.group(1))
    # prog-id uses hyphen in some logs
    m = re.search(r'prog-id=([0-9]+)', line)
    if m:
        numeric["prog_id"] = int(m.group(1))
    return numeric

def classify_container_threat(line):
    """
    Return tuple: (threat: 'benign'|'malicious', threat_type (string), attack_id or None)
    Rules are heuristics to bootstrap labels. Extend as needed.
    """
    low = line.lower()

    # Priority 1: container escape / host breakout indicators
    if re.search(r'\b(ptrace|nsenter|mount|pivot_root|chroot)\b', low):
        return "malicious", "Container Escape / Host Breakout", "C_ESC_001"

    # suspicious BPF loads by non-kernel/non-system processes
    if "type=bpf" in low and "op=load" in low:
        # check for kernel/system triggers
        if not re.search(r'kernel|auditd|systemd', low):
            return "malicious", "BPF / Kernel API Misuse", "C_BPF_002"

    # Privilege/capability abuse
    if re.search(r'\b(capset|capabilities|setuid|setgid)\b', low):
        return "malicious", "Privilege / Capability Abuse", "C_PRIV_003"
    if re.search(r'\bchmod\s+7+|chown\s+root\b', low):
        return "malicious", "Privilege / Capability Abuse", "C_PRIV_003"

    # Unauthorized exec / shell spawn inside container (common indicator)
    if re.search(r'execve.*(\/bin\/bash|\/bin\/sh|python|perl|/usr/bin/python|nc|netcat|socat)', low):
        return "malicious", "Unauthorized Exec / Shell Spawn", "C_EXEC_004"

    # Suspicious binary / miner detection
    for kw in SUSPICIOUS_BINARY_KEYWORDS:
        if kw in low and not any(w in low for w in ("systemd", "ssh")):
            return "malicious", "Suspicious Binary Activity (e.g., miner)", "C_MALBIN_005"

    # Secrets access
    if re.search(r'(/run/secrets|/etc/.*secret|\/proc\/[0-9]+\/environ|docker\.secrets|env\s*=\s*)', low):
        return "malicious", "Secrets Access", "C_SECRET_006"

    # File tampering / persistence (cron installs, writes to /etc)
    if re.search(r'\b(crontab|cron\.d|/etc/cron|/etc/init\.d|/etc/systemd)\b', low) or re.search(r'\b(open|write|creat).*(/etc|/root|/var/lib)\b', low):
        return "malicious", "File Tampering / Persistence", "C_PERSIST_007"

    # Supply-chain / unexpected package manager activity inside container
    if re.search(r'\b(apt-|yum|dnf|pip install|npm install|apk add)\b', low):
        return "malicious", "Supply-chain / Image Compromise Activity", "C_SUPPLY_008"

    # Network exfiltration / unusual networking by app processes
    if 'key="net_connect"' in line:
        comm = re.search(r'comm="([^"]+)"', line)
        if comm:
            proc = comm.group(1)
            if proc not in BENIGN_PROCS:
                return "malicious", "Malicious Networking / Exfiltration", "C_NET_009"
        else:
            return "malicious", "Malicious Networking / Exfiltration", "C_NET_009"

    # Default: mark known container runtime/system processes as benign
    comm = re.search(r'comm="([^"]+)"', line)
    if comm:
        if comm.group(1) in BENIGN_PROCS:
            return "benign", "Benign System Activity", None

    # If nothing matched, mark benign as default (we prefer fewer false positives on heuristics)
    return "benign", "Benign System Activity", None

def parse_audit_timestamp(line):
    """
    Parse timestamps from audit logs. Supports both:
    - Epoch form: audit(1762081984.956:7525)
    - Date form: audit(06/11/25 13:02:22.475:10774)
    Returns ISO8601 UTC string or None.
    """
    # Case 1: Epoch format (e.g., 1762081984.956)
    m_epoch = re.search(r'audit\((\d+\.\d+):\d+\)', line)
    if m_epoch:
        return epoch_to_iso(m_epoch.group(1))

    # Case 2: Human-readable date (e.g., 06/11/25 13:02:22.475)
    m_human = re.search(r'audit\((\d{2}\/\d{2}\/\d{2})\s+(\d{2}:\d{2}:\d{2}\.\d+):\d+\)', line)
    if m_human:
        date_str, time_str = m_human.groups()
        try:
            # interpret as YY/MM/DD HH:MM:SS.mmm
            dt = datetime.datetime.strptime(f"{date_str} {time_str}", "%y/%m/%d %H:%M:%S.%f")
            # ensure UTC (you can localize if needed)
            return dt.isoformat(timespec="milliseconds") + "Z"
        except ValueError:
            pass

    return None


def parse_audit_line_to_json(line):
    iso_ts = parse_audit_timestamp(line)
    comm_match = re.search(r'comm="([^"]+)"', line)
    entity_id = comm_match.group(1) if comm_match else "kernel"

    # event type selection
    key_match = re.search(r'key="([^"]+)"', line)
    op_match = re.search(r'op=([A-Za-z_]+)', line)
    type_match = re.search(r'type=([A-Za-z_]+)', line)
    event_type = key_match.group(1) if key_match else (op_match.group(1) if op_match else (type_match.group(1) if type_match else "UNKNOWN"))

    numeric = parse_numeric_fields(line)
    threat, threat_type, attack_id = classify_container_threat(line)

    return {
        "timestamp": iso_ts,
        "entity_id": entity_id,
        "layer": "container",
        "event_type": event_type,
        "message": line.strip(),
        "numeric": numeric,
        "labels": {
            "threat": threat,
            "threat_type": threat_type,
            **({"attack_id": attack_id} if attack_id else {})
        }
    }


def windows_from_events(events, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE):
    """
    Create sliding windows (lists of events). Each window inherits:
      - timestamp = last event timestamp
      - entity_id = entity with most events in window (mode)
      - labels: if any event in window is malicious, window labeled as that threat_type (choose majority or first)
    """
    if window_size <= 0:
        return []

    windows = []
    n = len(events)
    for start in range(0, max(1, n - window_size + 1), stride):
        w = events[start:start + window_size]
        if not w:
            continue
        # choose dominant entity_id by count
        ent_counts = defaultdict(int)
        for ev in w:
            ent_counts[ev.get("entity_id","unknown")] += 1
        dominant_entity = max(ent_counts.items(), key=lambda x: x[1])[0]
        # determine label: if any malicious, mark malicious; pick most common malicious threat_type
        malicious_events = [ev for ev in w if ev["labels"]["threat"] == "malicious"]
        if malicious_events:
            # pick most frequent threat_type among malicious events
            type_counts = defaultdict(int)
            for ev in malicious_events:
                type_counts[ev["labels"].get("threat_type","unknown")] += 1
            chosen_type = max(type_counts.items(), key=lambda x: x[1])[0]
            window_label = {"threat": "malicious", "threat_type": chosen_type}
        else:
            window_label = {"threat": "benign", "threat_type": "Benign System Activity"}

        windows.append({
            "start_idx": start,
            "end_idx": start + window_size - 1,
            "timestamp": w[-1]["timestamp"],
            "entity_id": dominant_entity,
            "events": w,
            "labels": window_label
        })
    return windows

# events = []
# with open(INPUT_FILE, "r", errors="ignore") as f:
#     for line in f:
#         line = line.strip()
#         if not line:
#             continue
#         parsed = parse_audit_line_to_json(line)
#         if parsed:
#             events.append(parsed)

# write per-event JSON
# with open(OUTPUT_FILE, "w") as out:
#     json.dump(events, out, indent=2)

# # produce windows if requested
# windows = windows_from_events(events, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE)
# if windows:
#     with open(WINDOW_OUTPUT_FILE, "w") as out_w:
#         json.dump(windows, out_w, indent=2)

# print(f"Processed {len(events)} events. Output: {OUTPUT_FILE}")
# if windows:
#     print(f"Produced {len(windows)} windows. Output: {WINDOW_OUTPUT_FILE}")



In [5]:

import re, datetime, json
from collections import defaultdict

def iso_now():
    return datetime.datetime.utcnow().isoformat(timespec='milliseconds') + 'Z'

# ---------- Layer parsers ----------

def is_noise_line(line: str) -> bool:
    # Filters out lines that aren't actual logs
    line = line.strip()
    return (
        not line
        or line.startswith("---")
        or line.startswith("===")
        or line.lower().startswith("log start")
        or len(line.split()) < 3  # too short, not a real log
    )



def parse_container_log(line):
    parsed = parse_audit_line_to_json(line)
    return parsed if parsed else None


def parse_network_log(line):
    # Example: 13:11:21.750116 IP 172.18.0.1.35746 > 172.18.0.5.http: Flags [S], seq ...
    m = re.match(r'(\d{2}:\d{2}:\d{2}\.\d+) IP ([\d\.]+)\.([0-9a-zA-Z]+) > ([\d\.]+)\.([0-9a-zA-Z]+): (.*)', line)
    if not m:
        return None
    ts, src_ip, src_port, dst_ip, dst_port, rest = m.groups()
    
    # quick heuristic labeling
    if any(flag in rest for flag in ["[S]", "[F]", "[R]"]) and "Flags" in rest:
        threat = "benign"
        threat_type = "Normal TCP handshake"
        attack_id = None
        if "nmap" in rest.lower() or "masscan" in rest.lower():
            threat, threat_type, attack_id = "malicious", "Network Scan", "N_SCAN_001"
        elif "http flood" in rest.lower():
            threat, threat_type, attack_id = "malicious", "HTTP Flood", "N_DOS_002"
    else:
        threat, threat_type, attack_id = "benign", "Normal Traffic", None
    
    return {
        "timestamp": iso_now(),  # approximate
        "entity_id": f"{src_ip}:{src_port}",
        "layer": "network",
        "event_type": "connection",
        "message": line.strip(),
        "numeric": {},
        "labels": {"threat": threat, "threat_type": threat_type, **({"attack_id": attack_id} if attack_id else {})},
        "relation": {"src": f"{src_ip}:{src_port}", "dst": f"{dst_ip}:{dst_port}"}
    }


def parse_application_log(line):
    # Typical microservice log line
    # payment-service-1 | timestamp="..." Method="POST" URL="/process" event="AuthenticationAttempt"
    m = re.search(r'timestamp="([^"]+)" .*?event="([^"]+)"', line)
    service_match = re.match(r'([a-zA-Z0-9\-_]+)\s*\|', line)
    service = service_match.group(1) if service_match else "unknown-service"
    timestamp = m.group(1) if m else iso_now()
    event_type = m.group(2) if m else "unknown"
    
    # heuristic threat labeling
    if "AuthenticationFailure" in line or "unauthorized" in line.lower():
        threat, threat_type, attack_id = "malicious", "Auth Brute Force", "A_AUTH_001"
    elif "SQL" in line and "error" in line.lower():
        threat, threat_type, attack_id = "malicious", "SQL Injection", "A_SQL_002"
    else:
        threat, threat_type, attack_id = "benign", "Normal Application Activity", None
    
    return {
        "timestamp": timestamp,
        "entity_id": service,
        "layer": "application",
        "event_type": event_type,
        "message": line.strip(),
        "numeric": {},
        "labels": {"threat": threat, "threat_type": threat_type, **({"attack_id": attack_id} if attack_id else {})}
    }



In [6]:


events = []
for path, parser in [
    ("container_logs.txt", parse_container_log),
    ("network_logs.txt", parse_network_log),
    ("application_logs.txt", parse_application_log)
]:
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if is_noise_line(line):
                continue
            parsed = parser(line)
            if parsed:
                events.append(parsed)


In [7]:
events[0]

{'timestamp': '2006-11-25T13:02:22.475Z',
 'entity_id': 'kernel',
 'layer': 'container',
 'event_type': 'PROCTITLE',
 'message': 'type=PROCTITLE msg=audit(06/11/25 13:02:22.475:10774) : proctitle=/usr/lib/systemd/systemd-resolved',
 'numeric': {},
 'labels': {'threat': 'benign', 'threat_type': 'Benign System Activity'}}

In [8]:
events[775]

{'timestamp': '2025-11-09T10:59:55.433Z',
 'entity_id': '172.18.0.1:48372',
 'layer': 'network',
 'event_type': 'connection',
 'message': '13:11:25.852735 IP 172.18.0.1.48372 > 172.18.0.5.http: Flags [P.], seq 1:201, ack 1, win 63, options [nop,nop,TS val 1929752186 ecr 1772366600], length 200: HTTP: POST /api/auth/register HTTP/1.1',
 'numeric': {},
 'labels': {'threat': 'benign', 'threat_type': 'Normal Traffic'},
 'relation': {'src': '172.18.0.1:48372', 'dst': '172.18.0.5:http'}}

In [9]:
events[-1]

{'timestamp': '2025-11-06T13:11:59.044+05:30',
 'entity_id': 'notification-service-1',
 'layer': 'application',
 'event_type': 'NotificationStored',
 'message': 'notification-service-1  | timestamp="2025-11-06T13:11:59.044+05:30" Method="POST" URL="/send" User-Agent="axios/1.13.2" Pragma="-" Cache-Control="-" Accept="application/json, text/plain, */*" Accept-encoding="gzip, compress, deflate, br" Accept-charset="-" language="-" host="notification-service:3003" cookie="-" content-type="application/json" connection="close" length="149" content="{"userId":1762414603673,"email":"charlie@paypal.com","type":"PAYMENT_SUCCESS","message":"Your payment of $150.75 to sarah@store.com was successful."}" event="NotificationStored" notificationId="1c2498ea-b12f-4338-af88-c8e7223c0088" userId="1762414603673"',
 'numeric': {},
 'labels': {'threat': 'benign', 'threat_type': 'Normal Application Activity'}}

In [10]:
len(events)

1359

In [11]:
for idx in range(len(events)):
    if events[idx]['layer']== "application":
        print(events[idx])
        break

{'timestamp': '2025-11-09T10:59:55.461Z', 'entity_id': 'unknown-service', 'layer': 'application', 'event_type': 'unknown', 'message': '📋 Showing application logs...', 'numeric': {}, 'labels': {'threat': 'benign', 'threat_type': 'Normal Application Activity'}}


In [12]:

import torch
from torch_geometric.data import Data
from collections import defaultdict
import numpy as np

def build_event_graph(events):
    """
    Construct a temporal event graph from your structured logs.
    Each event -> a node.
    Edges connect temporally adjacent or semantically related events.
    """
    num_nodes = len(events)

    # ----- NODE FEATURES -----
    # simple numerical + categorical encoding
    layers = {"k8s": 0, "container": 1, "network": 2, "application": 3}
    event_types = {}
    type_counter = 0

    node_features = []
    labels = []
    timestamps = []

    for ev in events:
        etype = ev.get("event_type", "UNKNOWN")
        if etype not in event_types:
            event_types[etype] = type_counter
            type_counter += 1

        feat = [
            layers.get(ev["layer"], -1),
            event_types[etype],
        ]
        node_features.append(feat)
        labels.append(1 if ev["labels"]["threat"] == "malicious" else 0)
        timestamps.append(ev.get("timestamp"))

    x = torch.tensor(node_features, dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.long)

    # ----- TEMPORAL EDGE CONSTRUCTION -----
    edge_index = []
    time_sorted = sorted(enumerate(timestamps), key=lambda t: (t[1] is None, t[1]))
    idx_order = [i for i, _ in time_sorted]

    # temporal adjacency edges
    for i in range(len(idx_order) - 1):
        src = idx_order[i]
        dst = idx_order[i + 1]
        edge_index.append([src, dst])
        edge_index.append([dst, src])  # bidirectional

    # intra-entity edges (same container/process)
    entity_map = defaultdict(list)
    for i, ev in enumerate(events):
        entity_map[ev["entity_id"]].append(i)

    for eid, idxs in entity_map.items():
        for i in range(len(idxs) - 1):
            edge_index.append([idxs[i], idxs[i + 1]])
            edge_index.append([idxs[i + 1], idxs[i]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).T  # shape [2, num_edges]

    # Build PyG graph
    data = Data(x=x, edge_index=edge_index, y=y)

    return data



c:\Users\tanis\anaconda3\envs\capstone\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
graph_data = build_event_graph(events)

In [14]:
graph_data

Data(x=[1359, 2], edge_index=[2, 5340], y=[1359])

In [15]:

from torch_geometric.nn import GCNConv
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# Training loop
def train_model(data):
    model = GCN(in_channels=data.x.size(1), hidden_channels=64, num_classes=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(50):
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            pred = out.argmax(dim=1)
            acc = (pred == data.y).float().mean()
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Acc: {acc.item():.4f}")

    return model

# Run

model = train_model(graph_data)



Epoch 0, Loss: 2.5774, Acc: 0.0942
Epoch 10, Loss: 0.4452, Acc: 0.9441
Epoch 20, Loss: 0.3824, Acc: 0.9441
Epoch 30, Loss: 0.2866, Acc: 0.9235
Epoch 40, Loss: 0.2231, Acc: 0.9419


In [16]:

# For each event pair (source, destination, timestamp)
# e.g. event i happens before event j at time t
edge_src = []
edge_dst = []
edge_t = []
edge_y = []  # labels per edge

for i in range(len(events) - 1):
    src = i
    dst = i + 1
    ts = events[dst].get("timestamp")
    if ts is None:
        continue
    edge_src.append(src)
    edge_dst.append(dst)
    edge_t.append(
        torch.tensor(
            np.datetime64(ts).astype('datetime64[ms]').astype(np.int64) / 1000.0
        )
    )
    edge_y.append(1 if events[dst]["labels"]["threat"] == "malicious" else 0)

edge_src = torch.tensor(edge_src)
edge_dst = torch.tensor(edge_dst)
edge_t = torch.tensor(edge_t, dtype=torch.float)
edge_y = torch.tensor(edge_y, dtype=torch.long)



C:\Users\tanis\AppData\Local\Temp\ipykernel_35500\3558179677.py:18: DeprecationWarning: parsing timezone aware datetimes is deprecated; this will raise an error in the future
  np.datetime64(ts).astype('datetime64[ms]').astype(np.int64) / 1000.0


In [ ]:
from torch_geometric.data import TemporalData
# temporal_tgn_from_events.py
import math, sys, time, datetime
from collections import defaultdict
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from torch.nn import Linear
from torch_geometric.loader import TemporalDataLoader
from torch_geometric.nn import TGNMemory, TransformerConv
from torch_geometric.nn.models.tgn import (
    IdentityMessage,
    LastAggregator,
    LastNeighborLoader,
)
import random


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------
# Helper: convert ISO timestamp to epoch seconds (float)
# -----------------------
def iso_to_epoch_seconds(iso_ts):
    if iso_ts is None:
        return None
    try:
        s = iso_ts
        # handle trailing Z
        if s.endswith("Z"):
            s = s[:-1] + "+00:00"
        dt = datetime.datetime.fromisoformat(s)
        return dt.timestamp()
    except Exception:
        # try parsing common formats
        try:
            # fallback: try yyyy-mm-ddTHH:MM:SS.mmm+HH:MM
            return datetime.datetime.strptime(iso_ts, "%Y-%m-%dT%H:%M:%S.%f%z").timestamp()
        except Exception:
            return None

# -----------------------
# 1) Build entity-id map (node ids)
# -----------------------
def build_entity_node_map(events):
    nodes = {}
    next_id = 0
    for ev in events:
        ent = ev.get("entity_id", "unknown")
        if ent not in nodes:
            nodes[ent] = next_id
            next_id += 1
        # also include network relation partners if present
        rel = ev.get("relation")
        if rel:
            s = rel.get("src")
            d = rel.get("dst")
            for p in (s, d):
                if p and p not in nodes:
                    nodes[p] = next_id
                    next_id += 1
    return nodes

# -----------------------
# 2) Build temporal interactions from events
# Strategy:
#   - Sort events by timestamp
#   - For consecutive events within a max_gap (e.g., 5s), create an interaction
#     src = entity(prev), dst = entity(curr)
#   - If event has explicit relation (network), also add that interaction (src_ip -> dst_ip)
# -----------------------
def build_interactions(events, node_map, max_gap=5.0):
    rows = []
    # convert timestamp to epoch and filter
    enriched = []
    for i, ev in enumerate(events):
        t = iso_to_epoch_seconds(ev.get("timestamp"))
        if t is None:
            continue
        enriched.append((t, i, ev))
    enriched.sort(key=lambda x: x[0])

    # consecutive interactions
    for idx in range(1, len(enriched)):
        t_prev, i_prev, ev_prev = enriched[idx - 1]
        t_curr, i_curr, ev_curr = enriched[idx]
        if (t_curr - t_prev) <= max_gap:
            src_ent = ev_prev.get("entity_id", "unknown")
            dst_ent = ev_curr.get("entity_id", "unknown")
            rows.append({
                "src_ent": src_ent,
                "dst_ent": dst_ent,
                "t": t_curr,
                "ev_src": ev_prev,
                "ev_dst": ev_curr,
            })

    # explicit relation edges from events (e.g., network src->dst)
    for (_, _, ev) in enriched:


SyntaxError: incomplete input (1726631943.py, line 99)

In [20]:
from torch_geometric.nn.models import TGNMemory
from torch_geometric.nn.models import TGN
from torch_geometric_temporal.nn.recurrent import TGNMemory
from torch_geometric_temporal.nn.models.tgn import TGN
import torch.nn as nn

num_nodes = len(events)
in_channels = 2  # (layer id, event type) — same as before
memory_dim = 64
time_dim = 64
embedding_dim = 64
output_dim = 2  # malicious/benign

memory = TGNMemory(
    num_nodes=num_nodes,
    raw_message_dim=in_channels,
    memory_dimension=memory_dim,
    time_dimension=time_dim
)

gnn = TGN(
    memory=memory,
    message_dimension=in_channels,
    memory_dimension=memory_dim,
    time_dimension=time_dim,
    embedding_dimension=embedding_dim,
    num_classes=output_dim
)

optimizer = torch.optim.Adam(gnn.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()



ImportError: cannot import name 'TGN' from 'torch_geometric.nn.models' (c:\Users\tanis\anaconda3\envs\capstone\lib\site-packages\torch_geometric\nn\models\__init__.py)